In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

FACT_ORDER_ITEMS = f"{CATALOG}.{SCHEMA_PREFIX}_gold.fact_order_items"
DIM_PRODUCT = f"{CATALOG}.{SCHEMA_PREFIX}_gold.dim_product"
AGG_DAILY_SALES = f"{CATALOG}.{SCHEMA_PREFIX}_gold.agg_daily_sales"

In [0]:
spark.sql(f"""
INSERT OVERWRITE {AGG_DAILY_SALES}
    (date_key, country_key, category, orders_count, items_count, quantity,
     gross_amount_usd, discount_amount_usd, net_amount_usd, processed_at)
SELECT
    f.date_key,
    f.country_key,
    p.category,
    COUNT(DISTINCT f.order_id),
    COUNT(*),
    SUM(f.quantity),
    CAST(SUM(f.gross_amount_usd)    AS DECIMAL(14,2)),
    CAST(SUM(f.discount_amount_usd) AS DECIMAL(14,2)),
    CAST(SUM(f.net_amount_usd)      AS DECIMAL(14,2)),
    current_timestamp()
FROM {FACT_ORDER_ITEMS} f
JOIN {DIM_PRODUCT} p ON f.product_id = p.product_id
GROUP BY f.date_key, f.country_key, p.category
""")